# Sparse Tensors and CNN-based Analysis

In this notebook, we'll walk through the process of converting simulated output to a MinkowskiEngine SparseTensor object, and show a simple sparse CNN-based regression model

In [1]:
from gampixpy import config, detector, generator, analysis, output

In [2]:
# set up matplotlib for this notebook
%matplotlib widget

# import torch and set the default device (needed for GPU)
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    # Set the default device to CUDA
    torch.set_default_device(device)
    print(f"Default device set to: {torch.cuda.get_device_name(device)}")
else:
    device = torch.device('cpu')
    print("CUDA is not available, using CPU")

import spconv.pytorch as spconv

CUDA is not available, using CPU


In [3]:
# the `demo_large_pixels` readout config is nice for visualizing
# but it will often not produce enough pixel hits.
# use whichever suits your purpose!
conf = config.ConfigManager(detector_config = 'far_detector_vd',
                            #readout_config = 'demo_large_pixels',
                            readout_config = 'GAMPixD',
                            )

## We can also build one of these from a yaml file
## The configs which come along with the library are
## accessible from the `preset_[...]_configs`, but can
## also be found alongside the library code
# readout_config = config.ReadoutConfig('../../gampixpy/readout_config/GAMPixD_notruth.yaml')

detector_model = detector.DetectorModel(config_manager=conf)

In [4]:
ps_gen = generator.PointSource(x_range = [-10, 10],
                               y_range = [-10, 10],
                               z_range = [-10, 10],
                               t_range = [0, 0],
                               q_range = [1e6, 1e8],
                              )

In [5]:
# clean the old output
!rm test_output.h5
# re-initialize the output manager
om = output.OutputManager('test_output.h5', config_manager=conf)

# here, we'll do 10 events
n_events = 10

# let's add a nice progress bar
import tqdm
for i in tqdm.tqdm(range(n_events)):
    # generate a new point source object
    event_data = ps_gen.get_sample()
    event_meta = ps_gen.get_meta()

    # use `verbose = False` to suppress the info messages
    detector_model.simulate(event_data, verbose = False)

    # write this to the output file
    om.add_entry(event_data, event_meta)

100%|██████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.59it/s]


In [10]:
stc = analysis.SparseTensorConverter('./test_output.h5')

depth = stc.get_vertex_depth(event_id=0)
st = stc.get_features_and_indices(event_id=0, label=0)
print (depth)
print (st)
print (st.indices)
print (st.features)

tensor(327.1641)
SparseConvTensor[shape=torch.Size([420, 4])]
tensor([[ 0,  0,  2,  1],
        [ 0,  0,  2,  2],
        [ 0,  0,  2,  3],
        ...,
        [ 0,  4,  4, 17],
        [ 0,  4,  4, 18],
        [ 0,  4,  4, 19]], dtype=torch.int32)
tensor([[-1050.7500,    -7.2500,  2039.4004,   -85.1809],
        [-1050.7500,    -7.2500,  2039.9004,    31.4075],
        [-1050.7500,    -7.2500,  2040.4004,     6.7894],
        ...,
        [-1048.7500,    -6.2500,  2047.4004,   -30.0194],
        [-1048.7500,    -6.2500,  2047.9004,    -2.6165],
        [-1048.7500,    -6.2500,  2048.4004,     7.2669]])


In [7]:
from torch import nn
class ExampleNet(nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.net = spconv.SparseSequential(
            spconv.SparseConv3d(4, 64, 3), # just like nn.Conv3d but don't support group
            nn.BatchNorm1d(64), # non-spatial layers can be used directly in SparseSequential.
            nn.ReLU(),
            spconv.SubMConv3d(64, 64, 3, indice_key="subm0"),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            # when use submanifold convolutions, their indices can be shared to save indices generation time.
            spconv.SubMConv3d(64, 64, 3, indice_key="subm0"),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            spconv.SparseConvTranspose3d(64, 64, 3, 2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            spconv.ToDense(), # convert spconv tensor to dense and convert it to NCHW format.
            nn.Conv3d(64, 64, 3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.shape = shape

    def forward(self, features, coors, batch_size):
        coors = coors.int() # unlike torch, this library only accept int coordinates.
        x = spconv.SparseConvTensor(features, coors, self.shape, batch_size)
        return self.net(x)# .dense()

In [8]:
net = ExampleNet([20, 20, 40])
print (st.features)
print (st.indices)
net(st.features, st.indices, 1)
#net.net(st)

tensor([[-1050.7500,    -7.2500,  2039.4004,   -85.1809],
        [-1050.7500,    -7.2500,  2039.9004,    31.4075],
        [-1050.7500,    -7.2500,  2040.4004,     6.7894],
        ...,
        [-1048.7500,    -6.2500,  2047.4004,   -30.0194],
        [-1048.7500,    -6.2500,  2047.9004,    -2.6165],
        [-1048.7500,    -6.2500,  2048.4004,     7.2669]])
tensor([[ 0,  0,  2,  1],
        [ 0,  0,  2,  2],
        [ 0,  0,  2,  3],
        ...,
        [ 0,  4,  4, 17],
        [ 0,  4,  4, 18],
        [ 0,  4,  4, 19]], dtype=torch.int32)


ValueError: expected 2D or 3D input (got 5D input)